In [1]:
library(dplyr)
library(Seurat)
library(ggplot2)
library(scDblFinder)
library(SingleCellExperiment)
library(patchwork)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.5 but the current
version is 1.7.6; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count



Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colColl

In [2]:
data_dir <- "../Data"
matrix_files <- list.files(data_dir, full.names=TRUE, pattern="_matrix\\.mtx$")
length(matrix_files)

[1] 27

In [ ]:
for(matrix_file in matrix_files)
{
    prefix <- sub("_matrix\\.mtx", "", basename(matrix_file))
    print(paste("Processing: ", prefix))

    #get the count matrix
    counts <- ReadMtx(
        mtx = file.path(data_dir, paste0(prefix, "_matrix.mtx")),
        cells = file.path(data_dir, paste0(prefix, "_barcodes.tsv")),
        features = file.path(data_dir, paste0(prefix, "_features.tsv")),
        feature.column = 2
    )

    #create seurat object
    obj <- CreateSeuratObject(
        counts = counts,
        project=prefix,
        min.cells = 3,
        min.features=200
    )

    obj$sample_id <- prefix

    #calculate the percentage of genes mapped to mitochondrial genes
    obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern="^MT-")

    #calculate the percentage of genes mapped to erythroid genes
    obj[["percent.eryth"]] <- PercentageFeatureSet(obj, features=c("HBA1", "HBA2", "HBB", "HBM", "ALAS2"))

    #QC filtering
    obj <- subset(obj, subset = nFeature_RNA>500 & nCount_RNA>1500 & percent.mt < 15 & percent.eryth < 1)

    #doublet detection and removal
    counts_mat <- GetAssayData(obj, assay = "RNA", layer = "counts")
    dbl <- scDblFinder(counts_mat)
    singlet_cells <- colnames(dbl)[dbl$scDblFinder.class == "singlet"]
    obj <- subset(obj, cells = singlet_cells)
    rm(counts_mat, dbl, singlet_cells)

    #Normalize the data
    obj <- NormalizeData(obj, normalization.method='LogNormalize', scale.factor=10000, verbose=FALSE)

    #Find HVGS
    obj <- FindVariableFeatures(obj, selection.method='vst', nfeatures=2000, verbose=FALSE)

    #save this data
    saveRDS(obj, file=file.path("../ProcessedData", paste0(prefix, ".rds")), compress=FALSE)
    print(paste("Finished processing: ", prefix))
    rm(obj)
    gc()
}

[1] "Processing:  GSM6204109_P01"


Assuming the input to be a matrix of counts or expected counts.

Creating ~3967 artificial doublets...

Dimensional reduction

Evaluating kNN...

Training model...

iter=0, 569 cells excluded from training.

iter=1, 569 cells excluded from training.

iter=2, 569 cells excluded from training.

Threshold found:0.344

278 (5.6%) doublets called



[1] "Finished processing:  GSM6204109_P01"
[1] "Processing:  GSM6204110_P02"


Warning message in PercentageFeatureSet(obj, features = c("HBA1", "HBA2", "HBB", :
“The following features are not found in the 'RNA' assay: 'HBM', 'ALAS2'”
Assuming the input to be a matrix of counts or expected counts.

Creating ~2818 artificial doublets...

Dimensional reduction

Evaluating kNN...

Training model...

iter=0, 509 cells excluded from training.

iter=1, 509 cells excluded from training.

iter=2, 509 cells excluded from training.

Threshold found:0.335

182 (5.2%) doublets called



[1] "Finished processing:  GSM6204110_P02"
[1] "Processing:  GSM6204111_P03"


...

[1] "Finished processing:  GSM6204133_P25"
[1] "Processing:  GSM6204134_P26"


Assuming the input to be a matrix of counts or expected counts.

Creating ~3429 artificial doublets...

Dimensional reduction

Evaluating kNN...

Training model...

iter=0, 579 cells excluded from training.

iter=1, 579 cells excluded from training.

iter=2, 579 cells excluded from training.

Threshold found:0.35

248 (5.8%) doublets called



[1] "Finished processing:  GSM6204134_P26"
[1] "Processing:  GSM6204135_P27"


Assuming the input to be a matrix of counts or expected counts.

Creating ~7100 artificial doublets...

Dimensional reduction

Evaluating kNN...

Training model...

iter=0, 1234 cells excluded from training.

iter=1, 1234 cells excluded from training.

iter=2, 1234 cells excluded from training.

Threshold found:0.375

776 (8.7%) doublets called



[1] "Finished processing:  GSM6204135_P27"
